In [3]:

# nos librairies
import pandas as pd
import matplotlib as plt

In [23]:
# affichons notre dataframe clean en fixant le type de la variable jour en datetimes
df = pd.read_csv("../data/covid_hospit_clean.csv", parse_dates=["jour"])
print(df.head())

  dep  sexe       jour  hosp  rea  HospConv  SSR_USLD  autres  rad  dc
0  01     0 2020-03-18     2    0       NaN       NaN     NaN    1   0
1  01     1 2020-03-18     1    0       NaN       NaN     NaN    1   0
2  01     2 2020-03-18     1    0       NaN       NaN     NaN    0   0
3  02     0 2020-03-18    41   10       NaN       NaN     NaN   18  11
4  02     1 2020-03-18    19    4       NaN       NaN     NaN   11   6


## KPI GLOGAUX

In [46]:
df.dtypes

dep                    str
sexe                 int64
jour        datetime64[us]
hosp                 int64
rea                  int64
HospConv           float64
SSR_USLD           float64
autres             float64
rad                  int64
dc                   int64
dtype: object

In [47]:
df.columns

Index(['dep', 'sexe', 'jour', 'hosp', 'rea', 'HospConv', 'SSR_USLD', 'autres',
       'rad', 'dc'],
      dtype='str')

### nous allons chercher les indicateurs pertinents

In [48]:
# nous allons afficher un descriptif de nos données

df[["hosp",	"rea","HospConv","SSR_USLD","autres","rad","dc"]].describe().round(2)

,hosp,rea,HospConv,SSR_USLD,autres,rad,dc
count,338245.00,338245.00,228140.00,228140.00,228140.00,338245.00,338245.00
mean,113.80,13.36,61.50,35.98,2.99,2817.62,534.74
std,166.75,28.40,83.83,52.00,5.83,4173.61,730.47
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,23.00,1.00,15.00,6.00,0.00,500.00,105.00
50%,58.00,4.00,34.00,18.00,1.00,1404.00,278.00
75%,134.00,13.00,73.00,44.00,3.00,3338.00,653.00
max,3281.00,855.00,1115.00,565.00,170.00,48210.00,6463.00


Nous observons qu'il y'a un très grand écart entre le min et le max des différentes variables.
Ce ne sont pas de valeurs abbérantes.
Ici les grands chiffres sont normaux car :

la pandémie a duré 3 ans ;
il y a 338 245 observations ;
les indicateurs sont cumulés quotidiennement.

Si nous prenons par exemple le descriptif de la variable (hosp)
le nombre d'hospitalisé par jour varie de 0 à 3281 personnes avec une moyenne de 113,80 par jour qui repésente le double de sa médiane 58.

La majorité des observations présentent un nombre d'hospitalisations relativement faible, tandis qu'un nombre limité d'observations affiche des niveaux très élevés, ce qui tire la moyenne vers le haut.

De même concernant le nombre de personnes en réanimation (rea), on a un effectif qui varie en 0 et 855 avec une moyenne de 13,36 ce qui traduit que la plupart du temps, les services de réanimation fonctionnaient à un niveau modéré. Certaines périodes ont connu une forte pression hospitalière.

Concernant les décès (dc), on a entre 0 et 6463 décès par jour avec une moyenne de 534.74 et une médiane de 278. ce qui traduit quelques épisodes exceptionnels qui augmentent fortement les valeurs observées.
 


In [ ]:
# nous pouvons oberver que les périodes qui a été les plus critiques dans les services hospitaliers en France était:
df.groupby("jour")["hosp"].sum().sort_values(ascending=False). head(10)

jour
2020-11-16    66605
2022-02-07    66417
2022-02-08    66233
2020-11-17    65964
2020-11-15    65759
2022-02-06    65562
2022-02-04    65492
2022-02-09    65297
2022-02-01    65287
2020-11-18    65285
Name: hosp, dtype: int64

On observe qu'il y'a eu une forte hospitalisation pendant les périodes de :
- Novembre 2020
- Février 2022

In [ ]:
# les départements qui ont été les plus touchés par la pandémie cette période de 2020 à 2023 sont:
df.groupby("dep")["hosp"].sum().sort_values(ascending=False). head(10)

dep
75    1967155
13    1752745
59    1655530
92    1654392
93    1635225
94    1340362
69    1204623
78    1044302
91     900666
95     833454
Name: hosp, dtype: int64

On observe que les départements les plus affectés étaient : 75, 13, 59, 92, 93 on peut remarquer une forte représentation des départements de la région île-de-france.


In [ ]:
# Nous filtrer et afficherons les résultats observer le dernier jour de la collecte des données dans l'ensemble des départements de la France
dernier_jour = df["jour"].max()

df[df["jour"] == dernier_jour][["hosp","rea","rad","dc"]].sum()

hosp      26065
rea        1413
rad     1719619
dc       271091
dtype: int64

In [61]:
df.groupby("jour")["dc"].max().tail()

jour
2023-03-27    6456
2023-03-28    6460
2023-03-29    6462
2023-03-30    6462
2023-03-31    6463
Name: dc, dtype: int64

Comme nous l'avons observer précédemment dans le fichier exploration que les cumuls hommes et femmes ont été construits indépendamment du cumul global.

Donc par exemple dc(sexe=0) n'est pas focément égale à dc(sexe=1) + dc(sexe=2) comme nous l'avons observé.


In [64]:
# nous allons réaliser 2 datasets à partir de notre df et afin d'avoir un avec sexe =0 et le second avec sexe=1 et 2
# Pour nos analyses globales, nous allons filtrer et obtenir df_total

df_total = df[df["sexe"]==0]
display(df_total.head())
display(df_total.tail())

,dep,sexe,jour,hosp,rea,HospConv,SSR_USLD,autres,rad,dc
0,01,0,2020-03-18,2,0,NaN,NaN,NaN,1,0
3,02,0,2020-03-18,41,10,NaN,NaN,NaN,18,11
6,03,0,2020-03-18,4,0,NaN,NaN,NaN,1,0
9,04,0,2020-03-18,3,1,NaN,NaN,NaN,2,0
12,05,0,2020-03-18,8,1,NaN,NaN,NaN,9,0


,dep,sexe,jour,hosp,rea,HospConv,SSR_USLD,autres,rad,dc
338231,972,0,2023-03-31,14,0,12.0,2.0,0.0,4669,1101
338234,973,0,2023-03-31,4,1,3.0,0.0,0.0,6330,410
338237,974,0,2023-03-31,27,5,14.0,8.0,0.0,8876,980
338240,976,0,2023-03-31,0,0,0.0,0.0,0.0,1766,163
338243,978,0,2023-03-31,0,0,0.0,0.0,0.0,0,0


In [68]:
# Pour les analyses par sexe nous allons filtrer et obtenir df_sex
df_sex = df[df["sexe"].isin([1,2])]
display(df_sex.head())
display(df_sex.tail())

,dep,sexe,jour,hosp,rea,HospConv,SSR_USLD,autres,rad,dc
1,01,1,2020-03-18,1,0,NaN,NaN,NaN,1,0
2,01,2,2020-03-18,1,0,NaN,NaN,NaN,0,0
4,02,1,2020-03-18,19,4,NaN,NaN,NaN,11,6
5,02,2,2020-03-18,22,6,NaN,NaN,NaN,7,5
7,03,1,2020-03-18,1,0,NaN,NaN,NaN,0,0


,dep,sexe,jour,hosp,rea,HospConv,SSR_USLD,autres,rad,dc
338238,974,1,2023-03-31,12,3,7.0,2.0,0.0,4177,544
338239,974,2,2023-03-31,15,2,7.0,6.0,0.0,4643,429
338241,976,1,2023-03-31,0,0,0.0,0.0,0.0,739,100
338242,976,2,2023-03-31,0,0,0.0,0.0,0.0,1002,61
338244,978,1,2023-03-31,0,0,0.0,0.0,0.0,0,0


In [75]:
print (f"df_total comptabilise {df_total.shape[0]} lignes et {df_total.shape[1]} colonnes , \n df_sexe comptabilise {df_sex.shape[0]} lignes et {df_sex.shape[1]} colonnes")


df_total comptabilise 113118 lignes et 10 colonnes , 
 df_sexe comptabilise 225127 lignes et 10 colonnes


In [ ]:
Nous pouvons voir que df_sexe représente environ le double de notre df_total.

In [76]:
# nous allons faire une copie de notre df_sex sur laquelle nous allons faire nos premières analyses 
df_sex_copy = df_sex.copy()
display(df_sex_copy.head())


,dep,sexe,jour,hosp,rea,HospConv,SSR_USLD,autres,rad,dc
1,01,1,2020-03-18,1,0,NaN,NaN,NaN,1,0
2,01,2,2020-03-18,1,0,NaN,NaN,NaN,0,0
4,02,1,2020-03-18,19,4,NaN,NaN,NaN,11,6
5,02,2,2020-03-18,22,6,NaN,NaN,NaN,7,5
7,03,1,2020-03-18,1,0,NaN,NaN,NaN,0,0


In [88]:
# le nombre d'hospitalisation moyen par sexe
display( "Moyenne:", df_sex_copy.groupby("sexe")["hosp"].mean())
display("Mediane:", df_sex_copy.groupby("sexe")["hosp"].median())

'Moyenne:'

sexe
1    84.637776
2    85.284343
Name: hosp, dtype: float64

'Mediane:'

sexe
1    46.0
2    47.0
Name: hosp, dtype: float64

On observe approximativement le même effectif donc il y'a autant de femmes que d'hommes qui sont hospitalisés suite au Covid pendant cette période.

In [92]:
# le nombre de réanimation moyen par sexe
display( "Moyenne:", df_sex_copy.groupby("sexe")["rea"].mean().round(2))

'Moyenne:'

sexe
1    13.52
2     6.44
Name: rea, dtype: float64

In [94]:
# le nombre de décès moyen par sexe
display( "Moyenne:", df_sex_copy.groupby("sexe")["dc"].mean().round(2))

'Moyenne:'

sexe
1    459.81
2    340.18
Name: dc, dtype: float64

Nous observons qu'il y'a environs 2 fois plus d'hommes que de femmes qui sont entrés en réanimation et il y'a plus d'hommes qui sont décédés que de femmes suite à cette pandémie.

In [ ]:
observons le comportement sur le dernier jour

In [112]:
#le dernier jour
dernier_jour = df_sex_copy["jour"].max()
# Le filtrer dans la variable jour
df_dernier_jour = df_sex_copy[df_sex_copy["jour"]==dernier_jour]
# filtrer certaines données du dernier jour en fonction du sexe

df_dernier_jour.groupby("sexe")[["hosp","rea","dc", "rad"]].sum()

,hosp,rea,dc,rad
sexe,,,,
1,5919,446,77630,422089
2,6997,258,57385,433693


Les hommes ont-ils été plus touchés que les femmes en termes d'hospitalisation, de réanimation et de décès ? et les retours à domicile qu'en est 'il?